# GPT, generative pre-training

- 《Improving Language Understanding by Generative Pre-Training》
- 利用自回归语言模型进行无监督预训练，训练目标是预测下一个词，这样就能直接使用海量原始文本；
- 采用 Transformer 架构，在捕捉序列模式和迁移能力方面优于以往模型，因此选定其作为基础结构。
- Decoder-only：
    - 分词与嵌入：文本首先通过分词器切分成 token，再经过嵌入层转换为词向量，并添加可学习的位置编码，以引入位置信息；
    - 掩码多头自注意力：使用带掩码的多头注意力机制，确保模型在预测每个 token 时只能参考其前文，保持自回归性质；
    - 残差连接与归一化：每个模块使用残差连接与层归一化，随后通过前馈神经网络处理，再次进行残差与归一化；
    - GPT-1 包含 12 层这样的解码器模块
- 预训练 + 微调

# BERT, Bidirectional Encoder Representations from Transformers

- 认为，GPT-1的注意力是单向的，每个token只能关注它前面token的信息，但对于NLP任务，双向注意力可以看到序列里所有token的信息，这样对于上下文理解会更加全面
- 输入 → BERT → 通用表示 → 任务头 → 输出
- Encoder-only：
    - Token Embedding + Segment Embedding + Position Embedding，按位相加构成序列最终的输入向量
        - token embedding，是每个token的词向量，代表每个token的原始含义，此时没有上下文信息；
        - segment embedding，用于区分每个token属于哪个序列的embedding，每个序列共享一个embedding；
        - position embedding，每个位置一个embedding，和transformer使用sin和cos函数生成固定位置编码不同，此处是可学习参数
    - 经过不同的分类头得到最终的输出，微调时只有分类头是全新的随机初始化

# GPT-2:

- 《Language Models are Unsupervised Multitask Learners》
- 为了解决GPT-1和BERT虽然引入了预训练机制，但在处理下游任务时仍然需要微调的局限
- **prompt**：输入内容由实际数据和自然语言提示组成，提示部分用以明确当前任务的目标
- 模型架构：
    - 层归一化的位置**pre-norm**：从每个模块的输出移至输入阶段，即先进行层归一化再执行注意力计算。这种调整有助于提高训练的稳定性。同时，在最后一个子层的注意力模块后新增了额外的层归一化操作，进一步提升了模型的稳定性。
      即从$Sublayer(x) + x \to LayerNorm$的post-norm改为了$LayerNorm(x) \to Sublayer \to+ x$的pre-norm
    - 残差连接初始化的优化：减小了残差连接的初始化参数，旨在更好地维持梯度的传播效率，防止梯度消失或爆炸问题
    - 词汇表扩展：扩大词汇表容量
    - 模型规模提升：参数规模扩大
- 任务类型不需要再被设计进模型结构里（比如添加分类头、设置分类数量），而是通过prompt以自然语言的方式传给模型。模型也不是输出任务可能结果的预测概率值，而是以自然语言的形式给出答案

# GPT-3

- 《Language Models are Few-Shot Learners》
- 仅通过prompt和少量示例，就完成复杂任务，无需调整模型参数
- 零样本zero-shot、单样本one-shot、少样本few-shot，没有梯度更新
- 模型结构：
    - 与GPT-2类似，但引入**稀疏注意力机制**，即每个 token 不再对所有前文 token 做注意力计算，只关注部分 token，从而提升效率，便于模型扩展


GPT-4

- 多模态

# Pre-norm & Post-norm

- Pre-norm:
$$x_{l+1} = x_l + F(LayerNorm(x_l))$$
x → LayerNorm → F(x) → residual add

- Post-norm:
$$x_{l+1} = LayerNorm(x_l + F(x_l))$$
x → F(x) → residual add → LayerNorm

- pre-norm求偏导可知，$\frac{\partial x_{l+1}}{\partial x_l}=1+…$，至少有一条路径不经过LayerNorm，深层稳定性好；
- 但，深层后，pre-norm效果变差，因为当足够深度时，t足够大，$x_{t+1}$和$x_t$相差很小，所以$F_{t+1}(LayerNorm(x_{t+1}))$和$F_{t+1}(LayerNorm(x_t))$差别很小，因此原本的t+1层实际就相当于是又做了一个t层，并没有实质的更新学习。

# Llama

- 《Open and Efficient Foundation Language Models》
- 应更关注推理阶段的计算成本；
- 增加训练数据规模比扩大模型参数更能提升性能；
- 模型架构：Decoder-only
    - pre-norm
    - RMSNorm替代LayerNorm，减少归一化成本。RMSNorm去除归一化中的减均值步骤及偏置项
    $$LayerNorm = \frac{x-E[x]}{\sqrt{Var(x)+\epsilon}}\cdot\gamma+\beta$$
    $$RMSNorm = \frac{x}{\sqrt{Mean(x^2)+\epsilon}}\cdot\gamma$$
    - 使用 SiLU 激活函数(Sigmoid Linear Unit)，提高模型表达能力
    $$SiLU = x \cdot sigmoid(x) = \frac{x}{1+e^{-x}}$$
- RoPE, Rotary Position Embedding, 旋转位置编码：注意力计算时$qk^T$之间的点积计算并没有考虑token的位置信息，通过旋转位置编码，在attention中添加位置信息
    - 通过sin和cos函数构成的旋转矩阵，对向量进行旋转。定义旋转矩阵$R(\theta)$：
    $$
    R(\theta)=\begin{bmatrix}
    cos(\theta) & -sin(\theta) \\
    sin(\theta) & cos(\theta)
    \end{bmatrix}
    $$
    - 任意向量x，可以用$xR(\theta)$表示对该向量进行逆时针旋转θ角度
    - 结合性：$R(\theta_1)R(\theta_2)=R(\theta_1+\theta_2)$
    - $R(\theta)^T=R(-\theta)$
    - 正交矩阵：$R(\theta)R(\theta)^T=R(\theta)^TR(\theta)=E$,故不会改变向量的模长，因此不会改变原模型的稳定
    - 通过旋转矩阵对两个向量进行编码来添加位置信息，对每个向量根据它们的位置索引（分别为m和n）进行旋转。如果对q应用旋转矩阵R(m)，对k应用旋转矩阵R(n)，然后再进行点积计算：
    $qR(m)\cdot (kR(n))^T=qR(m)R(n)^Tk^T=qR(m-n)k^T$
      是天然的相对位置编码，而且因为位置信息是一个连续函数，所以可外推
    - 拓展到高维：上面说的是2维旋转矩阵，拓展到高维，将维度两两一组进行旋转（任意两个特征一组都可以），每一组在它们两个特征组成的子空间内进行旋转
    $$
    R(\theta,m)=\begin{bmatrix}
    cos(m\theta_0) & -sin(m\theta_0) & 0 & 0 & \cdots & 0 & 0 \\
    sin(m\theta_0) & cos(m\theta_0) & 0 & 0 & \cdots & 0 & 0 \\
    0 & 0 & cos(m\theta_1) & -sin(m\theta_1) & \cdots & 0 & 0 \\
    0 & 0 & sin(m\theta_1) & cos(m\theta_1) & \cdots & 0 & 0 \\
    \vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
    0 & 0 & 0 & 0 & \cdots & cos(m\theta_{/frac{d}{2}-1}) & -sin(m\theta_{/frac{d}{2}-1}) \\
    0 & 0 & 0 & 0 & \cdots & sin(m\theta_{/frac{d}{2}-1}) & cos(m\theta_{/frac{d}{2}-1}) \\
    \end{bmatrix}
    $$
    \*其中，m代表该token在序列中的位置，$\theta_i=\frac{1}{10000^{\frac{2i}{d}}}$
    - 由于$R(\theta,m)$的稀疏性，所以一般不通过矩阵乘法进行计算：
    $$
    R(\theta,m)x=
    \begin{bmatrix}
    cos{m\theta_0} \\ cos{m\theta_0} \\
    cos{m\theta_1} \\ cos{m\theta_1} \\
    \vdots \\
    cos{m\theta_{\frac{d}{2}-1}} \\ cos{m\theta_{\frac{d}{2}-1}} \\
    \end{bmatrix}
    \otimes
    \begin{bmatrix}
    x_0 \\ x_1 \\ x_2 \\ x_3 \\ \vdots \\ x_{d-2} \\ x_{d-1} \\
    \end{bmatrix}
    +
    \begin{bmatrix}
    sin{m\theta_0} \\ sin{m\theta_0} \\
    sin{m\theta_1} \\ sin{m\theta_1} \\
    \vdots \\
    sin{m\theta_{\frac{d}{2}-1}} \\ sin{m\theta_{\frac{d}{2}-1}} \\
    \end{bmatrix}
    \otimes
    \begin{bmatrix}
    -x_1 \\ x_0 \\ -x_3 \\ x_2 \\ \vdots \\ -x_{d-1} \\ x_{d-2} \\
    \end{bmatrix}
    $$
    \* 其中，x是token的特征向量，$x_i$表示特征向量第i个位置的值

# Llama-2

- 训练流程：
    - 预训练，pretraining：使用大规模无标注文本进行自回归训练
    - 监督微调，supervised fine-tuning，SFT：使用问答对进行标注训练，更符合人类意图
    - 奖励模型训练，reward modeling：利用人类偏好数据，对同一问题的多个问答进行排序，分别训练安全奖励模型safety和有用性奖励模型helpfulness
    - 强化学习，reinforcement learning：基于奖励模型提供的信号优化输出质量
- 分组查询注意力机制：Group Query Attention, GQA
    - 对多头注意力里Q/K/V的头数关系进行重构，在尽量不损失效果的前提下大幅降低KV cache和推理成本。query被分组，例如每组两个，每组对应一个key和一个value。

# MOE, Mixture of Experts 混合专家模型
- MoE对原来的FeedForwardLayer全连接模块进行改造：
    - 一般的dense稠密大模型的全连接都是先升维到原来的4倍再降维回来；
    - MoELayer将原来的一个全连接模块拆分成多个小的全连接模块，由路由网络决定走哪个专家网络。router输出每个expert的权重概率，选取top-k进行稀疏激活，分别计算expert的输出后加权求和，得到对于这个token的最终输出（expert的选择是对于每个token分别进行的，不是对当前序列的）
- 专家负载均衡问题：可能造成大量的token都被少数几个expert处理，其他专家占用了参数但很难被激活
    - 负载均衡损失：希望每个专家被调用的频率是相等时损失最小。$$LOSS_{balance}=\sum^{N}_{i=1}(f_i)^2$$
    \* 其中，$f_i$表示第 i 个专家被调用的频率
    - 但由于$f_i$的选择是通过top_K获取的，这个操作并不是数值计算，不可微，无法计算梯度进行优化，所以一般采用$$LOSS_{balance}=\sum^{N}_{i=1}f_i*p_i$$
    \* 其中，$p_i$是一个batch内所有token对第 i 个专家的路由概率的平均值，由router进行softmax计算得到，代表的该专家的重要性（无论该专家是否被选中）

---

# DeepSeekMOE：
- 将专家进一步细分，从原来的N个专家变成2N个，每个专家网络参数量也为原来的一般，激活专家数从原来的2个变成4个，在前向传播时还能维持和原来网络同样的计算代价。每个专家学习到更加专精的东西，并且专家数增加导致可能的组合更多更灵活；
- 设置一个共享专家，学习通用的基础能力，保证每次都被激活

---

# DeepSeekV2：
- MLA, Multi-Latent Attention,多头潜在注意力：
    - 对token的特征向量先通过一个参数矩阵$W^{DKV}$进行向下压缩down转化，减小维度。然后只需要缓存这个降维后的kv压缩特征$C^{KV}$。在进行计算需要使用真实k和v向量时，再从 KV cache 中提取这个kv压缩向量，通过两个解压矩阵$W^{UK}$和$W^{UV}$向上投影up升维到原来的维度。
    $$
      \begin{aligned}
      attention &= softmax(\frac{QK^T}{\sqrt{d}})V  \\
      &= softmax(\frac{XW^Q(C^{KV}W^{UK})^T}{\sqrt{d}})V    \\
      &= softmax(\frac{X \underline{W^Q{W^{UK}}^T} {C^{KV}}^T}{\sqrt{d}})V
      \end{aligned}
    $$
    - 合并矩阵：上式中的$W^Q{W^{UK}}^T$可以合并为$W^{QUK}$，该矩阵可以提前计算好，不用在推理时额外进行对K的解压计算
- RoPE + MLA：
    - 第i个token的q向量和第j个token的k向量进行点积：
    $$
      \begin{aligned}
      q_iR_i \cdot (k_jR_j)^T &= h_iW^QR_i (c_j^{KV}W^{UK}R_j)^T  \\
      &= h_iW^QR_i R_j^T {W^{UK}}^T {c_j^{KV}}^T
      \end{aligned}
    $$
    由于引入了$R_i, R_j$所以不能像上面一样在推理时矩阵提前合并
    - 解决：给Q和K向量额外增加一些维度来表示位置信息
    - 对于Q向量，通过$W^{QR}$权重为每个头生成一些原始特征，然后通过RoPE增加位置信息，再将其拼接concat到每个注意力头的q向量；
    - 对于K向量，通过$W^{KR}$生成一个共享矩阵，然后通过RoPE增加位置信息，复制到多个头共享位置信息；
    - 最终处理后，由于拼接，q和k向量一样长，但他们和v向量不一样长，由于q和k进行点积计算后得到的是一个权重值，所以没有影响

---

# DeepSeek-V3
- 在router中：原V2是对各个expert的权重logits用softmax，让所有专家的权重和为1；因为最终只选择topK个专家，选择后对权重还要在做一次归一化，所以在V3中不适用softmax，改为对每个expert的权重的logits用sigmoid函数，然后对选择的topK个专家再进行归一化，作为最终的专家权重
- MOE负载均衡：去掉了针对batch的辅助负载均衡损失函数；改为在训练时在每个step监控每个专家的负载，如果负载过高就自动调整减小它对应的权重（该权重只影响router选择，在和最终专家输出相乘时还是采用原本的权重值）
- 增加针对序列的负载均衡损失函数，和上面针对batch的类似，用各个专家被选择的频率值乘以平均概率值再求和。主要是防止在序列内部专家的负载不均
- MTP, Multi-Token Prediction 多token预测：
    - 在最后一个transformer block中，设置多个blocks，第一个和原来一样用于预测下一个token，另外增加其他的头，它们也从L-1层接入token的特征，最后经过分类头后，分别预测后面第2个、第3个、……、第k个token。这里的分类头参数是共享的。在计算loss的时候，会分别考虑T2,T3,T4的交叉熵损失，其中T2损失权重更大，T3,T4的损失权重会小一些。
    - 通过 预测+验证 加速：用第一个头预测的下一个token T2，认为是最准确的，即最终的输出。但后面几个token不确信，让第一个头再去逐个验证，这里的验证是可以并行计算的。接受最长的正确序列，假如T4的最后一个预测在验证时给出了更好的答案，那么就采用第一个头给出的预测。
- DeepSeek MTP：
    - 原MTP的问题：跨多步的token不知道它前面的几个token，不利于模型收敛。比如，第四个头的预测，要在不知道它前面3个token的前提下预测出T4
    - 解决：给每个头传入额外的信息。输入T1，经过L层的transformer block（此处是L层，而非L-1层），输出经过分类头预测出T2。然后增加一个MTP头，输入为“第L层的token输出特征” 和 “T2这个token经过embedding后的特征”，对这两个特征向量分别RMSNorm归一化之后再concat拼接，此时由于拼接导致维度为原来的2倍，故再经过一个线性层映射为原来的token的特征维度，然后再经过一个transformer block之后经过一个分类头预测T3。
    - 其中embedding层的参数是共享的，分类头的参数也是共享的
    - 每个MTP头的输入都来自两部分，要预测$T_k$，将$T_{k-1}$的预测token和$T_{k-1}$的真实token进行归一化后拼接输入
    - 把原来多个独立的MTP头并行预测，变成了多个轻量transformer block串行预测
    - 相对于原MTP串行计算慢一点但效果更好；相比于一个token一个token计算，由于使用了轻量transformer block，比原来的每次都用full transformer block计算量更小